# Step 8 — Sales Prediction

In this step, we will use the best-performing machine learning model from Step 7 to generate sales predictions on unseen test-period data.

The Random Forest model was selected because it achieved the best validation and test performance.

The prediction output will be used later in Step 9 for inventory recommendation.

In [1]:
# Step 8.1: Import Required Libraries

import pandas as pd
import numpy as np

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Step 8.2: Load Prediction Data

train = pd.read_csv("../data/processed/train_features.csv")

train["Date"] = pd.to_datetime(train["Date"])

# Same chronological split used in Step 7
valid_end = train["Date"].quantile(0.85)

prediction_data = train[train["Date"] > valid_end].copy()

print("Prediction Data Shape:", prediction_data.shape)
print("Prediction Date Range:",
      prediction_data["Date"].min(),
      "to",
      prediction_data["Date"].max())

Prediction Data Shape: (151640, 29)
Prediction Date Range: 2015-03-18 00:00:00 to 2015-07-31 00:00:00


In [6]:
# Step 8.3: Prepare Prediction Features

features = [
    "Store",
    "DayOfWeek",
    "Open",
    "Promo",
    "SchoolHoliday",
    "Year",
    "Month",
    "Day",
    "WeekOfYear",
    "StoreType",
    "Assortment",
    "CompetitionDistance",
    "Promo2",
    "IsStateHoliday",
    "Lag_1_Sales",
    "Lag_7_Sales",
    "Lag_14_Sales",
    "Rolling_7_Sales",
    "Rolling_14_Sales",
    "Rolling_30_Sales"
]

X_prediction = prediction_data[features].copy()

# Encode categorical features
X_prediction = pd.get_dummies(
    X_prediction,
    columns=["StoreType", "Assortment"]
)

print("Prediction Features Shape:", X_prediction.shape)
print("Missing Values:", X_prediction.isna().sum().sum())


Prediction Features Shape: (151640, 25)
Missing Values: 408


In [7]:
# Step 8.3.1: Check Missing Values by Feature

missing_values = X_prediction.isna().sum()

print(missing_values[missing_values > 0])

CompetitionDistance    408
dtype: int64


In [8]:
# Step 8.3.2: Remove Rows With Missing Prediction Features

before_rows = len(X_prediction)

X_prediction = X_prediction.dropna()

after_rows = len(X_prediction)

print("Rows Before:", before_rows)
print("Rows After:", after_rows)
print("Rows Removed:", before_rows - after_rows)
print("Missing Values Remaining:", X_prediction.isna().sum().sum())

Rows Before: 151640
Rows After: 151232
Rows Removed: 408
Missing Values Remaining: 0


In [9]:
# Step 8.4: Prepare Training Data for Prediction Model

train_end = train["Date"].quantile(0.70)

model_train_data = train[
    train["Date"] <= train_end
].copy()

model_train_data = model_train_data.dropna(
    subset=features
)

X_model_train = model_train_data[features].copy()
y_model_train = model_train_data["Sales"]

X_model_train = pd.get_dummies(
    X_model_train,
    columns=["StoreType", "Assortment"]
)

print("Training Data Shape:", X_model_train.shape)
print("Target Shape:", y_model_train.shape)
print("Missing Values:", X_model_train.isna().sum().sum())

Training Data Shape: (677354, 25)
Target Shape: (677354,)
Missing Values: 0


In [12]:
# Step 8.5: Train Random Forest Model

from sklearn.ensemble import RandomForestRegressor

random_forest_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

random_forest_model.fit(
    X_model_train,
    y_model_train
)

print("Random Forest model trained successfully.")

Random Forest model trained successfully.


In [13]:
# Step 8.6: Generate Sales Predictions

predicted_sales = random_forest_model.predict(X_prediction)

print("Prediction Shape:", predicted_sales.shape)
print("Minimum Predicted Sales:", predicted_sales.min())
print("Maximum Predicted Sales:", predicted_sales.max())
print("Prediction Generated Successfully.")

Prediction Shape: (151232,)
Minimum Predicted Sales: 0.0
Maximum Predicted Sales: 33000.09
Prediction Generated Successfully.


In [14]:
# Step 8.7: Add Predictions

prediction_data = prediction_data.dropna(
    subset=features
).copy()

prediction_data["Predicted_Sales"] = predicted_sales

print("Prediction Data Shape:", prediction_data.shape)

print(
    prediction_data[
        ["Store", "Date", "Sales", "Predicted_Sales"]
    ].head(10)
)

Prediction Data Shape: (151232, 30)
     Store       Date  Sales  Predicted_Sales
806      1 2015-03-18   3858          5135.69
807      1 2015-03-19   4748          4377.90
808      1 2015-03-20   4057          4830.39
809      1 2015-03-21   3909          4573.79
810      1 2015-03-22      0             0.00
811      1 2015-03-23   3565          3916.61
812      1 2015-03-24   3547          3764.52
813      1 2015-03-25   3531          3712.46
814      1 2015-03-26   3932          3515.61
815      1 2015-03-27   4005          3847.76


In [15]:
# Step 8.8: Check Prediction Performance

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

mae = mean_absolute_error(
    prediction_data["Sales"],
    prediction_data["Predicted_Sales"]
)

rmse = np.sqrt(
    mean_squared_error(
        prediction_data["Sales"],
        prediction_data["Predicted_Sales"]
    )
)

r2 = r2_score(
    prediction_data["Sales"],
    prediction_data["Predicted_Sales"]
)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

MAE: 514.2000772323318
RMSE: 832.053242091887
R2: 0.9556413368664244


In [19]:
# Step 8.10: Create Clean Prediction Output

prediction_output = prediction_data[
    [
        "Store",
        "Date",
        "Sales",
        "Predicted_Sales"
    ]
].copy()

print("Prediction Output Shape:", prediction_output.shape)
print("Missing Values:", prediction_output.isna().sum().sum())

print(prediction_output.head(10))

Prediction Output Shape: (151232, 4)
Missing Values: 0
     Store       Date  Sales  Predicted_Sales
806      1 2015-03-18   3858          5135.69
807      1 2015-03-19   4748          4377.90
808      1 2015-03-20   4057          4830.39
809      1 2015-03-21   3909          4573.79
810      1 2015-03-22      0             0.00
811      1 2015-03-23   3565          3916.61
812      1 2015-03-24   3547          3764.52
813      1 2015-03-25   3531          3712.46
814      1 2015-03-26   3932          3515.61
815      1 2015-03-27   4005          3847.76


In [21]:
# Step 8.10: Check Missing Columns

missing = prediction_data.isna().sum()

print(missing[missing > 0])

CompetitionOpenSinceMonth    47736
CompetitionOpenSinceYear     47736
Promo2SinceWeek              73712
Promo2SinceYear              73712
PromoInterval                73712
dtype: int64


In [22]:
# Save Clean Prediction Output

prediction_output.to_csv(
    "../data/processed/sales_predictions.csv",
    index=False
)

print("Clean prediction file saved successfully.")

Clean prediction file saved successfully.


In [23]:
# Step 8.10: Final Verification

print("===== STEP 8 FINAL VERIFICATION =====")

print("Prediction Data Shape:", prediction_data.shape)

print("Prediction Columns:")
print(prediction_data.columns.tolist())

print("Missing Values:", prediction_data.isna().sum().sum())

print("Predicted Sales Min:", prediction_data["Predicted_Sales"].min())
print("Predicted Sales Max:", prediction_data["Predicted_Sales"].max())

print("Prediction File Saved Successfully.")

print("===== STEP 8 VERIFICATION COMPLETE =====")

===== STEP 8 FINAL VERIFICATION =====
Prediction Data Shape: (151232, 30)
Prediction Columns:
['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'Year', 'Month', 'Day', 'WeekOfYear', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval', 'IsStateHoliday', 'Lag_1_Sales', 'Lag_7_Sales', 'Lag_14_Sales', 'Rolling_7_Sales', 'Rolling_14_Sales', 'Rolling_30_Sales', 'Predicted_Sales']
Missing Values: 316608
Predicted Sales Min: 0.0
Predicted Sales Max: 33000.09
Prediction File Saved Successfully.
===== STEP 8 VERIFICATION COMPLETE =====
